# Reproducibility Notebook — HST Framework

This notebook accompanies the paper **"A Hybrid Sentiment–Trajectory AI
Framework for Candidate Selection: Beyond Static Evaluation"** (Hussain,
Nasir, and [Co-Author]). It provides a complete, cell-by-cell walkthrough
of the code, the synthetic-data generation process, and the analyses
underlying every table and figure reported in the paper.

It is intended for readers, reviewers, and other researchers who wish to
**verify, replicate, or extend** the reported results, rather than take
them on faith. Every number this notebook prints can be checked directly
against the corresponding table in the paper.

**Companion repository:** the full codebase, the real-data CSV file, and
raw output logs are available at
<https://github.com/nasirhussain92/hst-candidate-selection-framework>.

**Contents:**
1. Setup
2. The shape-feature extractor Φ(·) (paper §3.2, Eq. 2–3)
3. Synthetic candidate-data generator (paper §4)
4. General vs. favorable scenario — reproduces Table 1 (paper §5.1–5.2)
5. Raw-sequence neural benchmark — reproduces Table 2 (paper §5.3)
6. Real-data validation on the Campus Recruitment dataset — reproduces Tables 3 & 4 (paper §6–§7)
7. Figure 1 — the conceptual framework diagram

**To verify our results:** run every cell in order (`Runtime` → `Run
all`). Each result cell is followed by the exact figure it should
reproduce, so it can be checked against the published paper directly.
**To probe or extend our results:** each major section closes with one or
more suggested robustness checks that a reviewer or independent
researcher may wish to run.

**Note on AI-assisted development:** portions of this notebook's
underlying code were developed with the assistance of a generative-AI
tool (Claude, Anthropic), as disclosed in the paper's AI Disclosure
Statement. All code was reviewed, tested, and validated by the authors,
and the results reproduced here match those reported and independently
audited in the paper.


## 1. Setup

Google Colab already has `numpy`, `scipy`, `scikit-learn`, `pandas`, and
`matplotlib` pre-installed, so there is nothing to install. We just import
everything we'll need.

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, ttest_rel
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
print("Setup complete.")

## 2. The shape-feature extractor Φ(·)

This is the core building block of the framework (paper §3.2, Equations
2–3). It takes **any** ordered sequence of numbers — it makes no
difference whether that sequence is a candidate's performance across five
assessment stages, or their sentiment across four interview stages — and
reduces it to four descriptive statistics:

- **slope** — the overall trend (is the sequence rising or falling?)
- **volatility** — how much the sequence fluctuates (standard deviation)
- **recovery** — how well the sequence rebounds after a dip
- **consistency** — how stable the sequence is from one stage to the next

The same function is used for both the assessment-performance trajectory
**L** and the sentiment trajectory **S** in the paper precisely because it
makes no assumption about what the sequence represents, only that it is
an ordered list of numbers. The cell below runs the function on two toy
sequences as a sanity check before it is applied to the full dataset.

In [ ]:
def phi(V):
    """Shape-feature extractor: [slope, volatility, recovery, consistency]
    V: (n, n_stages) array. Sequence-agnostic -- no domain-specific assumptions."""
    n, T = V.shape
    stages = np.arange(T)
    slope = np.array([np.polyfit(stages, V[i], 1)[0] for i in range(n)])
    volatility = V.std(axis=1)
    diffs = np.diff(V, axis=1)
    recovery = np.array([
        np.sum(diffs[i][:-1] < 0) and np.mean(diffs[i][np.where(diffs[i][:-1] < 0)[0] + 1])
        if np.any(diffs[i][:-1] < 0) else 0.0
        for i in range(n)
    ])
    consistency = -np.mean(np.abs(diffs), axis=1)
    return np.stack([slope, volatility, recovery, consistency], axis=1)

# Quick sanity check: try it on a toy sequence
toy = np.array([[50, 55, 60, 65, 70],   # steadily improving
                 [70, 60, 50, 40, 30]])  # steadily declining
print("Toy shape features [slope, volatility, recovery, consistency]:")
print(phi(toy))

**Robustness check:** replace `toy` above with a sequence that dips in
the middle and recovers (e.g. `[70, 50, 40, 60, 75]`) and re-run the cell.
The `recovery` value should increase, confirming the function behaves as
specified in Equation 2–3 of the paper.

## 3. Synthetic candidate-data generator

The paper first validates the framework on **synthetic data with a known
ground truth** (paper §4) before turning to real data in Section 6 of
this notebook. Validating against a known ground truth is a necessary
first check on whether the framework can recover a signal that was
deliberately built into the data, prior to trusting it on noisier,
real-world data.

Each synthetic candidate has:
- a `true_quality` latent variable (the underlying ability the model is trying to recover)
- a `growth_true` latent variable (tendency to improve across stages)
- a `composure_true` latent variable (affective stability)
- an **assessment-performance trajectory L** (4 stages), constructed from these latents plus stage-level noise and an accommodation-related fatigue term
- a **sentiment trajectory S** (4 stages), constructed similarly
- an **outcome**, generated under either a "general" (ability-dominant) or a "favorable" (growth-weighted) specification, matching the two scenarios reported in the paper

In [ ]:
N_STAGES = 4

def generate_candidates(n, seed, favorable=False, fatigue_strength=0.35):
    rng = np.random.default_rng(seed)

    true_quality = rng.normal(0, 1, n)
    growth_true = rng.normal(0.15, 1, n)      # tendency to improve across stages
    composure_true = rng.normal(0, 1, n)      # affective stability latent

    # static features correlated with true ability, plus CQ/AQ-style scores
    resume_score = 0.6 * true_quality + rng.normal(0, 0.8, n)
    cq_score = 0.4 * true_quality + rng.normal(0, 0.9, n)
    aq_score = 0.3 * true_quality + rng.normal(0, 0.9, n)
    x_static = np.stack([resume_score, cq_score, aq_score], axis=1)

    # accommodation-need flag: unrelated to true_quality by construction
    accommodation_need = rng.binomial(1, 0.15, n)

    # --- assessment performance trajectory L ---
    L = np.zeros((n, N_STAGES))
    for j in range(N_STAGES):
        stage_frac = j / (N_STAGES - 1)
        growth_component = growth_true * stage_frac
        fatigue = fatigue_strength * stage_frac * (1 + 0.8 * accommodation_need)
        noise = rng.normal(0, 0.5, n)
        L[:, j] = true_quality + growth_component - fatigue + noise

    # --- sentiment trajectory S ---
    S = np.zeros((n, N_STAGES))
    for j in range(N_STAGES):
        stage_frac = j / (N_STAGES - 1)
        recovery_wobble = 0.3 * np.sin(stage_frac * np.pi) * (composure_true > 0)
        noise = rng.normal(0, 0.5, n)
        S[:, j] = composure_true + recovery_wobble + noise

    # --- outcome ---
    if favorable:
        outcome_latent = 0.4 * true_quality + 0.3 * growth_true + 0.3 * composure_true
    else:
        outcome_latent = 0.85 * true_quality + 0.15 * growth_true

    realized_outcome = outcome_latent + rng.normal(0, 0.6, n)  # noisy real-world label

    return dict(
        x_static=x_static, L=L, S=S,
        true_quality=true_quality, growth_true=growth_true,
        composure_true=composure_true, accommodation_need=accommodation_need,
        outcome_latent=outcome_latent, realized_outcome=realized_outcome,
    )

# Generate one batch and peek at it
sample = generate_candidates(4000, seed=1, favorable=False)
print("Static features shape:", sample['x_static'].shape)
print("L (performance trajectory) shape:", sample['L'].shape)
print("First candidate's L sequence:", sample['L'][0])
print("First candidate's S sequence:", sample['S'][0])

**Robustness check:** set `fatigue_strength = 0.0` and regenerate the
data. This removes the accommodation-related fatigue mechanism entirely —
the same mechanism examined in the paper's fairness audit (§7) — and is a
useful check for readers wishing to isolate that effect from the rest of
the model.

## 4. General vs. favorable scenario (reproduces Table 1)

Two models are compared on the synthetic data:
- **static-only**, which predicts the outcome from static features alone
- **fusion**, which predicts the outcome from static features plus Φ(L) and Φ(S)

Both use ridge regression (a standard, regularized form of linear
regression). The comparison is repeated across 10 random train/test
splits, and a paired t-test — a standard statistical test for whether two
related sets of scores differ significantly — is used to assess the
result.

In [ ]:
def fit_eval(x_train, y_train, x_test, y_test):
    model = Ridge(alpha=1.0)
    model.fit(x_train, y_train)
    pred = model.predict(x_test)
    r, _ = pearsonr(pred, y_test)
    return r, model

def run_seed(data, seed, target_key="outcome_latent"):
    rng = np.random.default_rng(seed + 9999)
    n = data["x_static"].shape[0]
    idx = rng.permutation(n)
    split = int(n * 0.7)
    tr, te = idx[:split], idx[split:]

    y = data[target_key]
    x_static = data["x_static"]
    phi_L = phi(data["L"])
    phi_S = phi(data["S"])

    x_fusion = np.concatenate([x_static, phi_L, phi_S], axis=1)

    r_static, _ = fit_eval(x_static[tr], y[tr], x_static[te], y[te])
    r_fusion, fusion_model = fit_eval(x_fusion[tr], y[tr], x_fusion[te], y[te])

    return r_static, r_fusion, (x_fusion, fusion_model, te)

def multiseed_compare(data, n_seeds=10, target_key="outcome_latent"):
    statics, fusions = [], []
    last_fusion_bundle = None
    for s in range(n_seeds):
        r_s, r_f, bundle = run_seed(data, s, target_key)
        statics.append(r_s)
        fusions.append(r_f)
        last_fusion_bundle = bundle
    statics = np.array(statics)
    fusions = np.array(fusions)
    t, p = ttest_rel(fusions, statics)
    return dict(
        static_mean=statics.mean(), static_sd=statics.std(),
        fusion_mean=fusions.mean(), fusion_sd=fusions.std(),
        t=t, p=p, last_fusion_bundle=last_fusion_bundle,
    )

def report(name, res):
    print(f"\n=== {name} ===")
    print(f"static-only  r = {res['static_mean']:.4f} (sd {res['static_sd']:.4f})")
    print(f"fusion       r = {res['fusion_mean']:.4f} (sd {res['fusion_sd']:.4f})")
    print(f"paired t-test: t={res['t']:.4f}, p={res['p']:.6f}")

In [ ]:
N = 4000

# General scenario
data_general = generate_candidates(N, seed=1, favorable=False)
res_general_latent = multiseed_compare(data_general, target_key="outcome_latent")
report("GENERAL scenario vs true (latent) outcome", res_general_latent)
res_general_real = multiseed_compare(data_general, target_key="realized_outcome")
report("GENERAL scenario vs noisy realized outcome", res_general_real)

# Favorable (growth-weighted) scenario
data_fav = generate_candidates(N, seed=2, favorable=True)
res_fav_latent = multiseed_compare(data_fav, target_key="outcome_latent")
report("FAVORABLE (growth-weighted) scenario vs true (latent) outcome", res_fav_latent)
res_fav_real = multiseed_compare(data_fav, target_key="realized_outcome")
report("FAVORABLE (growth-weighted) scenario vs noisy realized outcome", res_fav_real)

**Reference values (paper Table 1):** in the general scenario, fusion
should offer only a small advantage over static-only scoring. In the
favorable (growth-weighted) scenario, the gap should widen substantially
and become highly significant (r ≈ 0.46 → 0.62, p < 0.0001). Readers
verifying this notebook should compare their printed output directly
against Table 1 in the published paper.

**Robustness check:** the outcome-generating weights inside
`generate_candidates` (the line defining `outcome_latent` for the
favorable scenario) can be adjusted to test whether the fusion advantage
scales as expected when growth is weighted even more heavily in the
outcome.

## 5. Raw-sequence neural benchmark (reproduces Table 2)

This section addresses a natural question: what if, instead of
hand-engineering Φ(·), a neural network is given the raw, unprocessed
sequences directly? A small multilayer perceptron (MLP) — one of the
simplest neural-network architectures — is used here as an approximation
of the cross-attention model proposed but not implemented in the paper
(Equation 5), for the reasons given in §3.5 and §5.3.

In [ ]:
def run_seed_with_neural(data, seed, target_key="outcome_latent"):
    rng = np.random.default_rng(seed + 9999)
    n = data["x_static"].shape[0]
    idx = rng.permutation(n)
    split = int(n * 0.7)
    tr, te = idx[:split], idx[split:]
    y = data[target_key]

    x_static = data["x_static"]
    phi_L, phi_S = phi(data["L"]), phi(data["S"])
    x_fusion = np.concatenate([x_static, phi_L, phi_S], axis=1)
    x_raw = np.concatenate([x_static, data["L"], data["S"]], axis=1)  # raw, unengineered

    r_static, _ = fit_eval(x_static[tr], y[tr], x_static[te], y[te])
    r_fusion, _ = fit_eval(x_fusion[tr], y[tr], x_fusion[te], y[te])

    mlp = MLPRegressor(hidden_layer_sizes=(32, 16), max_iter=2000,
                        random_state=seed, early_stopping=True)
    mlp.fit(x_raw[tr], y[tr])
    pred = mlp.predict(x_raw[te])
    r_neural, _ = pearsonr(pred, y[te])

    return r_static, r_fusion, r_neural

def compare_with_neural(data, n_seeds=10, target_key="outcome_latent"):
    S, F, Nn = [], [], []
    for s in range(n_seeds):
        rs, rf, rn = run_seed_with_neural(data, s, target_key)
        S.append(rs); F.append(rf); Nn.append(rn)
    S, F, Nn = np.array(S), np.array(F), np.array(Nn)
    t_fn, p_fn = ttest_rel(F, Nn)
    print(f"static  r mean={S.mean():.4f} sd={S.std():.4f}")
    print(f"fusion  r mean={F.mean():.4f} sd={F.std():.4f}")
    print(f"neural  r mean={Nn.mean():.4f} sd={Nn.std():.4f}")
    print(f"fusion vs neural: t={t_fn:.4f} p={p_fn:.6f}")

print("=== GENERAL scenario ===")
compare_with_neural(data_general, target_key="outcome_latent")

print("\n=== FAVORABLE scenario ===")
compare_with_neural(data_fav, target_key="outcome_latent")

**Reference values (paper Table 2):** the raw-sequence neural network
should substantially outperform fusion (r ≈ 0.96–0.97 vs. 0.62–0.69). This
is reported in the paper as an honest limitation rather than smoothed
over: Φ(·) discards a sequence's absolute level by design, and a raw
model is able to exploit that discarded information. Fusion remains
interpretable by construction; the raw model does not. Readers should
treat any deviation from these reference values as a signal to check
random seeds and package versions against `requirements.txt` in the
companion repository.

## 6. Real-data validation on Kaggle data (reproduces Tables 3 & 4)

This section moves from synthetic to **real** data: the Campus
Recruitment dataset (Roshan, n.d.; CC0 license), which records five
genuine sequential evaluation stages per student (10th-grade percentage →
12th-grade percentage → undergraduate degree percentage → employability-test
percentage → MBA percentage) together with a real placement outcome.

**To run this section, upload the CSV first.** Execute the cell below,
then use the "Choose Files" prompt to upload
`Placement_Data_Full_Class.csv` — available in the `data/` folder of the
companion GitHub repository, or directly from
<https://www.kaggle.com/datasets/benroshan/factors-affecting-campus-placement>.

In [ ]:
from google.colab import files
uploaded = files.upload()  # select Placement_Data_Full_Class.csv when prompted

In [ ]:
df = pd.read_csv("Placement_Data_Full_Class.csv")
df.columns = [c.strip() for c in df.columns]

stage_cols = ["ssc_p", "hsc_p", "degree_p", "etest_p", "mba_p"]
L_real = df[stage_cols].to_numpy(dtype=float)
y_status = (df["status"].str.strip() == "Placed").astype(int).to_numpy()

print(f"N = {len(df)}, placed = {y_status.sum()}, not placed = {(1-y_status).sum()}")
df[stage_cols].describe().loc[["mean", "std", "min", "max"]]

In [ ]:
phi_L_real = phi(L_real)
x_static_real = np.stack([L_real[:, -1], L_real.mean(axis=1)], axis=1)  # [mba_p, mean_of_all_stages]
x_fusion_real = np.concatenate([x_static_real, phi_L_real], axis=1)

def cv_auc(X, y, n_splits=5, n_repeats=20):
    aucs = []
    for rep in range(n_repeats):
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=rep)
        for tr, te in skf.split(X, y):
            scaler = StandardScaler().fit(X[tr])
            Xtr, Xte = scaler.transform(X[tr]), scaler.transform(X[te])
            clf = LogisticRegression(max_iter=1000)
            clf.fit(Xtr, y[tr])
            p = clf.predict_proba(Xte)[:, 1]
            if len(np.unique(y[te])) == 2:
                aucs.append(roc_auc_score(y[te], p))
    return np.array(aucs)

auc_static = cv_auc(x_static_real, y_status)
auc_fusion = cv_auc(x_fusion_real, y_status)
n_match = min(len(auc_static), len(auc_fusion))
t_auc, p_auc = ttest_rel(auc_fusion[:n_match], auc_static[:n_match])

print("=== Predicting placement status (5-fold CV x 20 repeats, AUC) ===")
print(f"static-only  AUC = {auc_static.mean():.4f} (sd {auc_static.std():.4f})")
print(f"traj-fusion  AUC = {auc_fusion.mean():.4f} (sd {auc_fusion.std():.4f})")
print(f"paired t-test: t={t_auc:.4f}, p={p_auc:.6f}")

In [ ]:
# Salary among placed candidates
placed_mask = y_status == 1
Ls = L_real[placed_mask]
salary = df.loc[placed_mask, "salary"].to_numpy(dtype=float)
phi_Ls = phi(Ls)
x_static_s = np.stack([Ls[:, -1], Ls.mean(axis=1)], axis=1)
x_fusion_s = np.concatenate([x_static_s, phi_Ls], axis=1)

def cv_corr(X, y, n_splits=5, n_repeats=20):
    corrs = []
    for rep in range(n_repeats):
        kf = KFold(n_splits=n_splits, shuffle=True, random_state=rep)
        for tr, te in kf.split(X):
            scaler = StandardScaler().fit(X[tr])
            Xtr, Xte = scaler.transform(X[tr]), scaler.transform(X[te])
            reg = Ridge(alpha=1.0)
            reg.fit(Xtr, y[tr])
            pred = reg.predict(Xte)
            if np.std(pred) > 1e-8:
                r, _ = pearsonr(pred, y[te])
                corrs.append(r)
    return np.array(corrs)

corr_static = cv_corr(x_static_s, salary)
corr_fusion = cv_corr(x_fusion_s, salary)
n2 = min(len(corr_static), len(corr_fusion))
t2, p2 = ttest_rel(corr_fusion[:n2], corr_static[:n2])

print(f"=== Predicting salary among placed (N={placed_mask.sum()}), 5-fold CV x 20 repeats ===")
print(f"static-only  r = {corr_static.mean():.4f} (sd {corr_static.std():.4f})")
print(f"traj-fusion  r = {corr_fusion.mean():.4f} (sd {corr_fusion.std():.4f})")
print(f"paired t-test: t={t2:.4f}, p={p2:.6f}")

**Reference values (paper Table 3):** fusion should clearly outperform
static-only scoring for *placement* (AUC 0.869 → 0.923, p < 0.0001), but
slightly underperform static-only scoring for *salary* (r 0.163 → 0.126,
p = 0.006). This asymmetry is reported in the paper as a genuine finding:
trajectory shape assists in predicting selection but not compensation,
and readers should not expect it to disappear on re-running this cell.

### Fairness audit (reproduces Table 4)

This section checks whether the trajectory feature behaves differently
between placed and non-placed candidates in a way that is relevant to the
fairness mechanism examined on synthetic data in the paper's fairness
audit (§7). On the real dataset, this is assessed via a straightforward
mechanism check: comparing the shape-feature means between outcome
groups.

In [ ]:
print("=== Shape-feature means by outcome group ===")
labels = ["slope", "volatility", "recovery", "consistency"]
for k, lab in enumerate(labels):
    print(f"  {lab}: placed={phi_L_real[y_status==1, k].mean():.4f}  "
          f"not_placed={phi_L_real[y_status==0, k].mean():.4f}")

## 7. Figure 1 — the conceptual framework diagram

This final section regenerates Figure 1 from the paper: the architecture
diagram showing how static features, the assessment-performance
trajectory, and the sentiment trajectory pass through Φ(·) and are fused
into the final HCS score. The output should be visually identical to
Figure 1 as published.

In [ ]:
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(11, 6.5))
ax.set_xlim(0, 11)
ax.set_ylim(0, 6.5)
ax.axis('off')

def box(x, y, w, h, text, fc="#f2f2f2", ec="#333333", fontsize=10, weight="normal", ls="-"):
    b = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.08",
                        linewidth=1.4, edgecolor=ec, facecolor=fc, linestyle=ls)
    ax.add_patch(b)
    ax.text(x + w/2, y + h/2, text, ha="center", va="center",
            fontsize=fontsize, weight=weight, wrap=True)

def arrow(x1, y1, x2, y2, style="-|>", color="#333333", lw=1.4, ls="-"):
    a = FancyArrowPatch((x1, y1), (x2, y2), arrowstyle=style,
                         mutation_scale=14, linewidth=lw, color=color, linestyle=ls)
    ax.add_patch(a)

box(0.3, 4.7, 2.1, 1.1, "Static Features\n$x_{static}$\n(resume, CQ, AQ)\nEq. 7", fc="#eaf2fb")
box(0.3, 2.9, 2.1, 1.1, "Assessment-Performance\nTrajectory  L\n(multi-stage scores)\n\u00a73.3", fc="#eaf7ef")
box(0.3, 1.1, 2.1, 1.1, "Sentiment Trajectory\nS\n(affect across interview\nstages)   Eq. 1 / 6", fc="#fdf1e6")
box(3.4, 2.0, 2.3, 2.0, "Shape-Feature\nExtractor  \u03a6(\u00b7)\n[slope, volatility,\nrecovery, consistency]\nEq. 2\u20133", fc="#ffffff", fontsize=9.5)
box(6.7, 2.0, 2.2, 2.0, "Linear Fusion\nEq. 4\n\n(implemented &\nempirically tested)", fc="#fff6e8", fontsize=9.5)
box(9.6, 2.6, 1.1, 0.9, "HCS\nscore", fc="#eaf7ef", weight="bold")
box(6.7, 4.6, 2.2, 1.1, "Proposed Cross-Attention\nFusion (Eq. 5)\n\u2014 not implemented \u2014", fc="#f7f7f7", ec="#999999", fontsize=8.7, ls="--")

arrow(2.4, 3.45, 3.4, 3.0)
arrow(2.4, 1.65, 3.4, 2.4)
arrow(2.4, 5.25, 6.7, 3.6, color="#4472c4")
arrow(5.7, 3.0, 6.7, 3.0)
arrow(8.9, 3.0, 9.6, 3.05)
arrow(2.4, 5.15, 6.7, 5.1, ls="--", color="#999999")
arrow(5.7, 3.6, 6.7, 4.9, ls="--", color="#999999")

ax.text(5.5, 6.15, "Figure 1. Hybrid Sentiment\u2013Trajectory (HST) Framework: Model Architecture",
        ha="center", fontsize=12, weight="bold")
plt.tight_layout()
plt.show()

## Summary

Running this notebook end to end reproduces every table and the figure
reported in the paper, from the underlying code and data rather than from
the reported numbers alone. Independent researchers wishing to build on
this work, or reviewers wishing to stress-test it beyond the checks
already reported, may find the following useful starting points:

- Vary `N` (sample size) in Section 4 and examine the stability of the reported effects.
- Vary the number of seeds / cross-validation repeats and examine how the standard deviations change.
- In Section 6, remove the `mba_p` stage from `stage_cols` and examine whether a four-stage trajectory changes the placement AUC materially.

Questions, replication issues, or extensions can be raised via the Issues
tab of the companion GitHub repository:
<https://github.com/nasirhussain92/hst-candidate-selection-framework>.
